#Week 3-4 Report for Titanic Project

####Richard Lin

This is my week 3-4 report for the Titanic project from Kaggle(https://www.kaggle.com/c/titanic).
In this report I simplied the data preprocessing process using python functions and applied a few different models to make the prediction.

##Data preprocessing

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Define the function to clean up data:

In [70]:
def clean(df):
    df['Gender'] = df['Sex'].map( {'female': 0, 'male': 1} ).astype(int)
    median_ages = np.zeros((2,3)) 
    for i in range(0, 2):
        for j in range(0, 3):
            median_ages[i,j] = df[(df['Gender'] == i) & (df['Pclass'] == j+1)]['Age'].dropna().median()
    df['AgeFill'] = df['Age']
    for i in range(0, 2):
        for j in range(0, 3):
            df.loc[ (df.Age.isnull()) & (df.Gender == i) & (df.Pclass == j+1), 'AgeFill'] = median_ages[i,j]
    df['FamilySize'] = df['SibSp'] + df['Parch']
    df = df.drop(['PassengerId','Name','Sex','Ticket','Fare','Cabin','Embarked','Age','SibSp','Parch','FamilySize'],axis=1)
    return df

Read data into data sets:

In [71]:
train_raw = pd.read_csv('data/train.csv', header=0)
test_raw =pd.read_csv('data/test.csv', header =0)

Clean the both data sets using the clean function:

In [72]:
train = clean(train_raw)
test = clean(test_raw)

In [73]:
x_train = train.ix[:,'Pclass':]
y_train = train.ix[:,'Survived']

In [74]:
x_test = test.ix[:,'Pclass':]

##Modelling
####Random Forests

In [75]:
from sklearn.ensemble import RandomForestClassifier 
forest = RandomForestClassifier(n_estimators = 100)
forest = forest.fit(x_train,y_train)

In [76]:
output_train = forest.predict(x_train)

In [77]:
result_train = pd.Series(output_train==y_train)

In [78]:
accuracy_train = result_train.value_counts()[True]*1.0/(result_train.value_counts()[True]+result_train.value_counts()[False])

In [79]:
accuracy_train

0.87991021324354657

In [80]:
test_raw['Survived'] = forest.predict(x_test)

In [81]:
test_raw[['PassengerId','Survived']].to_csv('RandomForest.csv', index = False)

The submission of the result got a score of 0.71770, which is not good at all.

####Logistic Regression

In [84]:
from sklearn import linear_model
logistic = linear_model.LogisticRegression(C=0.2)
logistic = logistic.fit(x_train,y_train)

In [85]:
test_raw['Survived'] = logistic.predict(x_test)

In [86]:
test_raw[['PassengerId','Survived']].to_csv('Logistic.csv', index = False)

The submission of the result got a score of 0.76555, which is much better than the score got using Random Forests.